# 04 · Train / validation / test / production splits
**AAI-540 · Group 4 · Criteo CTR**

| Split | Share | Time window (by row position) | Used for |
|---|---|---|---|
| train | 40% | earliest | fitting models **and** the categorical encoder |
| validation | 10% | next | early stopping, hyperparameter tuning |
| test | 10% | next | one-time final evaluation |
| production | 40% | most recent | simulated live traffic: batch inference + monitoring (labels held back) |

**Why chronological, not random:** a deployed CTR model always predicts *future* traffic. A random split lets
same-hour campaigns and sessions appear on both sides, inflating validation scores and hiding the drift the
monitoring system exists to catch. Production is deliberately the most recent 40%.

Source of truth is the **Feature Store offline store** (not the raw sample), so training data is exactly what
was registered. All logic lives in `src/criteo_ctr/datasets.py` and is unit-tested.

In [ ]:
import os, sys, json, time, datetime as dt
sys.path.insert(0, os.path.abspath("../src"))

import boto3
import sagemaker
import pandas as pd
from botocore.exceptions import ClientError

from criteo_ctr import config as C
from criteo_ctr import features as F
from criteo_ctr.athena import Athena
from criteo_ctr.io_utils import Store, s3_uri, lower_columns, canonical_columns, default_data_dir

sess = sagemaker.Session()
boto_sess = sess.boto_session
region = sess.boto_region_name
role = sagemaker.get_execution_role()
bucket = C.BUCKET or sess.default_bucket()
store = Store(bucket, boto_sess)
athena = Athena(boto_sess, s3_uri(bucket, C.ATHENA_RESULTS_PREFIX) + "/", database=C.ATHENA_DATABASE)

print(f"region={region}\nbucket={bucket}\nprefix={C.PREFIX}\nathena db={C.ATHENA_DATABASE}")

In [ ]:
from sagemaker.feature_store.feature_group import FeatureGroup
from criteo_ctr import datasets as D

tables = {}
for name in (C.FG_NUMERIC, C.FG_CATEGORICAL, C.FG_LABEL):
    q = FeatureGroup(name=name, sagemaker_session=sess).athena_query()
    tables[name] = q.table_name
    FS_DB = q.database
tables

## 1 · Join the three offline tables (latest version of each record)

In [ ]:
def latest(table, cols):
    return (f"SELECT {cols} FROM (SELECT *, row_number() OVER (PARTITION BY record_id "
            f"ORDER BY write_time DESC, api_invocation_time DESC) AS rn "
            f'FROM "{table}" WHERE NOT is_deleted) WHERE rn = 1')

num_sel = ", ".join(c.lower() for c in D.META_COLS + D.NUMERIC_FS_COLS)
cat_sel = ", ".join(["record_id"] + [c.lower() for c in D.HASH_COLS])
cat_out = ", ".join(f"c.{c.lower()}" for c in D.HASH_COLS)

STAGING = f"{C.PREFIX}/staging/fs_join/"
store.delete_prefix(STAGING)            # UNLOAD requires an empty target
t0 = time.time()
qid = athena.execute(f"""
UNLOAD (
  SELECT n.*, {cat_out}, l.label
  FROM ({latest(tables[C.FG_NUMERIC], num_sel)}) n
  JOIN ({latest(tables[C.FG_CATEGORICAL], cat_sel)}) c ON n.record_id = c.record_id
  JOIN ({latest(tables[C.FG_LABEL], "record_id, label")}) l ON n.record_id = l.record_id
) TO '{s3_uri(bucket, STAGING)}' WITH (format = 'PARQUET', compression = 'SNAPPY')
""", database=FS_DB)
print(f"UNLOAD done in {time.time()-t0:.0f}s, scanned {athena.scanned_mb(qid):,.0f} MB")

joined = store.read_parquet_prefix(STAGING)
print(f"{len(joined):,} joined records x {joined.shape[1]} columns")

## 2 · Split, fit the encoder on train only, validate

In [ ]:
t0 = time.time()
result = D.build_model_datasets(joined)
checks = D.validate_model_datasets(result)
print(f"built in {time.time()-t0:.1f}s")
for k, v in checks.items():
    print(("PASS " if v else "FAIL ") + k)

m = result.manifest
split_table = pd.DataFrame(m["splits"]).T[["rows", "fraction", "ctr", "record_id_min", "record_id_max",
                                           "day_index_min", "day_index_max"]]
split_table

In [ ]:
import matplotlib.pyplot as plt
FIG_DIR = os.path.abspath("../reports/figures"); os.makedirs(FIG_DIR, exist_ok=True)
colors = {"train": "#66c2a5", "validation": "#8da0cb", "test": "#e78ac3", "production": "#fc8d62"}

fig, axes = plt.subplots(1, 2, figsize=(13, 2.8), gridspec_kw={"width_ratios": [2.2, 1]})
left = 0
for name in C.SPLIT_ORDER:
    frac = m["splits"][name]["fraction"]
    axes[0].barh(0, frac, left=left, color=colors[name])
    axes[0].text(left + frac / 2, 0, f"{name}\n{frac:.0%}", ha="center", va="center", fontsize=9)
    left += frac
axes[0].set_xlim(0, 1); axes[0].set_yticks([]); axes[0].set_xlabel("position in time (share of records) ->")
axes[0].set_title("Chronological split")
axes[1].bar(C.SPLIT_ORDER, [m["splits"][s]["ctr"] for s in C.SPLIT_ORDER], color=[colors[s] for s in C.SPLIT_ORDER])
axes[1].set_title("CTR by split"); axes[1].tick_params(axis="x", rotation=20)
plt.tight_layout(); plt.savefig(os.path.join(FIG_DIR, "11_splits.png"), bbox_inches="tight")
store.upload_file(os.path.join(FIG_DIR, "11_splits.png"), f"{C.PREFIX}/reports/figures/11_splits.png")
plt.show()

prod_unseen = {c: float((result.splits["production"][f"{c}_enc"] == C.RARE_ID).mean()) for c in C.CAT_COLS}
print("production rows mapped to RARE/unseen, worst 5:",
      dict(sorted(prod_unseen.items(), key=lambda kv: -kv[1])[:5]))

## 3 · Write the splits
* `splits/xgboost/{train,validation,test}/*.csv` — SageMaker built-in XGBoost format (label first, no header)
* `splits/xgboost/production/production_features.csv` — **no label**, input for Batch Transform
* `splits/ground_truth/production_labels.parquet` — held back for Model Monitor model-quality jobs
* `splits/parquet/split=*/` — full frames with `record_id`, cataloged in Athena as `model_dataset`
* `artifacts/` — fitted encoder, feature column order, split manifest

In [ ]:
store.delete_prefix(f"{C.SPLITS_PREFIX}/")
out = {}
for name, frame in result.splits.items():
    out[f"parquet/{name}"] = store.put_parquet(lower_columns(frame), f"{C.SPLITS_PREFIX}/parquet/split={name}/part-00000.parquet")
for name in ("train", "validation", "test"):
    out[f"xgboost/{name}"] = store.put_text(D.to_xgboost_csv(result.splits[name]),
                                            f"{C.SPLITS_PREFIX}/xgboost/{name}/{name}.csv")
prod = result.splits["production"]
out["xgboost/production_features"] = store.put_text(D.to_xgboost_csv(prod, include_label=False),
                                                    f"{C.SPLITS_PREFIX}/xgboost/production/production_features.csv")
out["ground_truth/production_labels"] = store.put_parquet(prod[[C.RECORD_ID, C.EVENT_TIME, C.LABEL]],
                                                          f"{C.SPLITS_PREFIX}/ground_truth/production_labels.parquet")

out["encoder"] = store.put_text(result.encoder.to_json(), f"{C.ARTIFACTS_PREFIX}/categorical_encoder.json")
out["feature_columns"] = store.put_json(F.model_feature_columns(), f"{C.ARTIFACTS_PREFIX}/feature_columns.json")
m["validation_checks"] = checks
m["created_utc"] = dt.datetime.utcnow().isoformat() + "Z"
m["outputs"] = out
out["manifest"] = store.put_json(m, f"{C.ARTIFACTS_PREFIX}/split_manifest.json")
pd.Series(out, name="s3_uri").to_frame()

## 4 · Catalog the model dataset in Athena and reconcile

In [ ]:
cols = lower_columns(result.splits["train"]).dtypes
ddl_cols = ",\n  ".join(f"{c} {'bigint' if str(t).startswith('int') else 'double'}" for c, t in cols.items())
athena.execute("DROP TABLE IF EXISTS model_dataset")
athena.execute(f"""
CREATE EXTERNAL TABLE model_dataset (
  {ddl_cols}
)
COMMENT 'Encoded, chronologically split model dataset (40/10/10/40)'
PARTITIONED BY (split string)
STORED AS PARQUET
LOCATION '{s3_uri(bucket, C.SPLITS_PREFIX)}/parquet/'
""")
athena.execute("MSCK REPAIR TABLE model_dataset")
recon = athena.query("""
SELECT split, count(*) AS rows, avg(label) AS ctr, min(record_id) AS first_id, max(record_id) AS last_id
FROM model_dataset GROUP BY split
ORDER BY CASE split WHEN 'train' THEN 1 WHEN 'validation' THEN 2 WHEN 'test' THEN 3 ELSE 4 END""")
ok = all(int(r.rows) == m["splits"][r.split]["rows"] for r in recon.itertuples())
print("Athena counts reconcile with manifest:", "PASS" if ok else "FAIL")
assert ok
recon